# 00 — EDA on NASA C-MAPSS turbofan engine degradation

This is the first notebook in the TurboGuard pipeline. We:

1. Load the four C-MAPSS sub-datasets.
2. Inspect engine lifetime distributions.
3. Look at sensor degradation trajectories.
4. Identify constant / non-informative sensors.
5. Save a quick sanity baseline (mean cycle as the predicted RUL) for benchmarking later models.

**Pre-req:** run `bash scripts/download_data.sh` first.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from turboguard.data.cmapss import DATASETS, load_cmapss, add_rul_clipped
from turboguard.models.rul.nasa_score import nasa_score

## 1. Load all four sub-datasets

FD001 is the simplest (single operating condition, single fault mode); FD004 is the hardest (six op conditions, two fault modes).

In [ ]:
data = {name: load_cmapss(name) for name in DATASETS}

summary = pd.DataFrame({
    name: {
        'train_engines': d.train['unit_id'].nunique(),
        'test_engines': d.test['unit_id'].nunique(),
        'train_rows': len(d.train),
        'test_rows': len(d.test),
        'mean_cycles_train': d.train.groupby('unit_id')['cycle'].max().mean(),
    }
    for name, d in data.items()
}).T
summary

## 2. Engine lifetime distributions

How long do engines run before failure?

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharey=True)
for ax, (name, d) in zip(axes, data.items()):
    lifetimes = d.train.groupby('unit_id')['cycle'].max()
    ax.hist(lifetimes, bins=20)
    ax.set_title(f'{name} — lifetimes (cycles)')
    ax.set_xlabel('cycles to failure')
axes[0].set_ylabel('engines')
fig.tight_layout()

## 3. Sensor trajectories for a few engines (FD001)

Most papers focus on a subset of the 21 sensors that show clear monotonic degradation. We'll plot several and identify which ones are informative vs. constant.

In [ ]:
fd001 = data['FD001'].with_rul()
sensor_cols = [c for c in fd001.columns if c.startswith('sensor_')]

fig, axes = plt.subplots(7, 3, figsize=(14, 14), sharex=False)
for ax, col in zip(axes.flat, sensor_cols):
    for uid in [1, 50, 100]:
        sub = fd001[fd001.unit_id == uid]
        ax.plot(sub.cycle, sub[col], alpha=0.6, label=f'unit {uid}')
    ax.set_title(col, fontsize=9)
fig.tight_layout()

## 4. Constant / non-informative sensors

Sensors with zero (or near-zero) variance across an engine's life carry no predictive signal — drop them before modeling.

In [ ]:
stds = fd001.groupby('unit_id')[sensor_cols].std().mean()
constant_sensors = stds[stds < 1e-6].index.tolist()
print(f'Sensors with ~zero variance in FD001: {constant_sensors}')
stds.sort_values(ascending=False).head(10)

## 5. Naive baseline — predict mean cycle as RUL

Bottom-of-the-barrel benchmark: the NASA score for predicting the train-set mean lifetime as the RUL of every test engine. Anything we model later should beat this.

In [ ]:
fd001_d = data['FD001']
mean_lifetime = fd001_d.train.groupby('unit_id')['cycle'].max().mean()

# At the last test cycle, the predicted RUL is (mean_lifetime - last_cycle).
last_cycles = fd001_d.test.groupby('unit_id')['cycle'].max().values
y_pred = np.maximum(mean_lifetime - last_cycles, 0)
y_true = fd001_d.rul['RUL'].values

print(f'NASA score (FD001, mean-baseline): {nasa_score(y_true, y_pred):,.0f}')
print(f'RMSE                              : {np.sqrt(((y_pred - y_true)**2).mean()):.2f}')

---

## Next notebooks

- `01_features.ipynb` — rolling stats, frequency-domain features, change-point detection.
- `02_baselines_xgb_lgbm.ipynb` — XGBoost / LightGBM RUL with MLflow tracking.
- `03_lstm_rul.ipynb` — sliding-window LSTM in PyTorch.
- `04_survival_weibull_cox.ipynb` — Weibull AFT and Cox PH on engine-level features.
- `05_anomaly.ipynb` — Isolation Forest + LSTM autoencoder on sensor streams.